![image](https://raw.githubusercontent.com/IBM/watson-machine-learning-samples/master/cloud/notebooks/headers/watsonx-Prompt_Lab-Notebook.png)
# Use AutoAI RAG and Milvus database to work with `ibm-watsonx-ai` SDK documentation.

#### Disclaimers

- Use only Spaces that are available in the watsonx context.


## Notebook content

This notebook contains the steps and code to demonstrate the usage of IBM AutoAI RAG. The AutoAI RAG experiment conducted in this notebook uses data scraped from the `ibm-watsonx-ai` SDK documentation.

Some familiarity with Python is helpful. This notebook uses Python 3.12.


## Learning goal

The learning goals of this notebook are:

- Create an AutoAI RAG job that will find the best RAG pattern based on provided data


## Contents

This notebook contains the following parts:

1. [Set up the environment](#Set-up-the-environment)
2. [RAG Optimizer definition](#RAG-Optimizer-definition)
3. [Run the RAG Experiment](#Run-the-RAG-Experiment)
4. [Comparison and testing of RAG Patterns](#Comparison-and-testing-of-RAG-Patterns)
5. [Historical runs](#Historical-runs)
6. [Cleanup](#Cleanup)
7. [Summary and next steps](#Summary-and-next-steps)

<a id="Set-up-the-environment"></a>
## Set up the environment

Before you use the sample code in this notebook, you must perform the following setup task:

-  Contact your IBM Cloud Pak® for Data administrator and ask them for your account credentials

### Install dependencies
**Note:** `ibm-watsonx-ai` documentation can be found <a href="https://ibm.github.io/watsonx-ai-python-sdk/index.html" target="_blank" rel="noopener no referrer">here</a>.

In [1]:
%pip install -U wget | tail -n 1
%pip install -U "ibm-watsonx-ai[rag]>=1.4.10" | tail -n 1

#### Define credentials

Authenticate the watsonx.ai Runtime service on IBM Cloud Pak® for Data. You need to provide the **admin's** `username` and the platform `url`.

In [2]:
import os

try:
    username = os.environ["USERNAME"]
except KeyError:
    username = input("Please enter your username (hit enter): ")

try:
    url = os.environ["URL"]
except KeyError:
    url = input("Please enter the platform url (hit enter): ")

Use the **admin's** `api_key` to authenticate watsonx.ai Runtime services:

In [3]:
import getpass

from ibm_watsonx_ai import Credentials

credentials = Credentials(
    username=username,
    api_key=getpass.getpass("Enter your watsonx.ai API key and hit enter: "),
    url=url,
    instance_id="openshift",
    version="5.4",
)

Alternatively you can use the **admin's** `password`:

In [4]:
import getpass

from ibm_watsonx_ai import Credentials

if "credentials" not in locals() or not credentials.api_key:
    credentials = Credentials(
        username=username,
        password=getpass.getpass("Enter your watsonx.ai password and hit enter: "),
        url=url,
        instance_id="openshift",
        version="5.4",
    )

#### Create `APIClient` instance

In [5]:
from ibm_watsonx_ai import APIClient

client = APIClient(credentials)

### Working with spaces

First, you need to create a space for your work. If you do not have a space already created, you can use `{PLATFORM_URL}/ml-runtime/spaces?context=icp4data` to create one.

- Click **New Deployment Space**
- Create an empty space
- Go to the space `Settings` tab
- Copy the `space_id` and paste it below

**Tip**: You can also use SDK to prepare the space for your work. Find more information in the [Space Management sample notebook](https://github.com/IBM/watson-machine-learning-samples/blob/master/cpd5.0/notebooks/python_sdk/instance-management/Space%20management.ipynb).

**Action**: Assign the space ID below

In [6]:
try:
    space_id = os.environ["SPACE_ID"]
except KeyError:
    space_id = input("Please enter your space_id (hit enter): ")

To print all existing spaces, use the `list` method.

In [7]:
client.spaces.list(limit=10)

,ID,NAME,CREATED
0,7467e11a-2c3c-485a-a170-1ef8a512f168,Auto-Created-RAG-space-1785319738.5778759-clea...,2026-07-29T10:08:59.387Z


To be able to interact with all resources available in watsonx.ai, you need to set the **space** which you will be using.

In [8]:
client.set.default_space(space_id)

'SUCCESS'

<a id="RAG-Optimizer-definition"></a>
## RAG Optimizer definition

### Define a connection to the training data

Define connection information to access the COS bucket and the file that contains the training data. This example uses [`ibm_watsonx_ai`](https://ibm.github.io/watsonx-ai-python-sdk/index.html) SDK documentation content.

The following code cell downloads the `ibm_watsonx_ai` Python SDK compressed file from GitHub (if not already downloaded), and extracts its contents to a specified folder.

In [9]:
import os
import zipfile

import wget

archive_name = "watsonx-ai-python-sdk"
archive_zip = "watsonx-ai-python-sdk.zip"

if not os.path.isfile(archive_zip):
    wget.download(
        "https://github.com/IBM/watsonx-ai-python-sdk/archive/refs/heads/gh-pages.zip",
        out=archive_zip,
    )

with zipfile.ZipFile(archive_zip, "r") as zip_ref:
    zip_ref.extractall(archive_name)

Create a connection to COS.

In [10]:
datasource_name = "bluemixcloudobjectstorage"

# Provide COS credentials
bucket_name = "PASTE YOUR BUCKET NAME HERE"
access_key = "PASTE YOUR ACCESS KEY HERE"
secret_key = "PASTE YOUR SECRET KEY HERE"
url = "PASTE YOUR URL HERE"

In [11]:
conn_meta_props = {
    client.connections.ConfigurationMetaNames.NAME: f"Connection to Database - {datasource_name} ",
    client.connections.ConfigurationMetaNames.DATASOURCE_TYPE: client.connections.get_datasource_type_id_by_name(
        datasource_name
    ),
    client.connections.ConfigurationMetaNames.DESCRIPTION: "Connection to external Database",
    client.connections.ConfigurationMetaNames.PROPERTIES: {
        "bucket": bucket_name,
        "access_key": access_key,
        "secret_key": secret_key,
        "iam_url": "https://iam.cloud.ibm.com/identity/token",
        "url": url,
    },
}

conn_details = client.connections.create(meta_props=conn_meta_props)
connection_id = client.connections.get_id(conn_details)

Creating connections...
SUCCESS


Create a Data Connection that represents input data references.

In [12]:
from ibm_watsonx_ai.helpers import DataConnection, S3Location

data_connection = DataConnection(
    connection_asset_id=connection_id,
    location=S3Location(bucket=bucket_name, path=archive_name),
)
input_data_references = [data_connection]
input_data_references[0].set_client(client)

Filter documents with the `.html` extension and save them to the COS bucket.

In [13]:
html_docs_files = []

for root, dirs, files in os.walk(archive_name):
    if root != f"{archive_name}/watsonx-ai-python-sdk-gh-pages/v1.4.11":
        continue

    for file in filter(lambda x: x.endswith(".html"), files):
        file_path = os.path.join(root, file)
        html_docs_files.append(file_path)

Writing all SDK documents might take around 3 minutes.

In [14]:
for i, html_docs_file in enumerate(html_docs_files):
    data_connection.write(html_docs_file, remote_name=html_docs_file.split("/")[-1])
    print(
        f"Progress: {'✓' * (i+1)}{'.' * (len(html_docs_files)-i-1)}",
        end="\r",
        flush=True,
    )

### Define a connection to the test data

Upload a `json` file that you want to use as a benchmark to COS and then define a connection to the file. This example uses content from the [`ibm_watsonx_ai`](https://ibm.github.io/watsonx-ai-python-sdk/index.html) SDK documentation.

In [15]:
benchmarking_data_IBM_page_content = [
    {
        "question": "How to install ibm-watsonx-ai library?",
        "correct_answer": "pip install ibm-watsonx-ai",
        "correct_answer_document_ids": ["install.html"],
    },
    {
        "question": "What is Credentials class parameters?",
        "correct_answer": "url, api_key, name, iam_serviceid_crn, token, projects_token, username, password, instance_id, version, bedrock_url, proxies, verify",
        "correct_answer_document_ids": ["base.html"],
    },
    {
        "question": "How to get AutoAI pipeline with number 3?",
        "correct_answer": "get_pipeline(pipeline_name='Pipeline_3')",
        "correct_answer_document_ids": ["autoai_working_with_class_and_optimizer.html"],
    },
    {
        "question": "How to get list of Embedding Models?",
        "correct_answer": "client.foundation_models.EmbeddingModels",
        "correct_answer_document_ids": ["fm_embeddings.html"],
    },
    {
        "question": "How to retrieve the list of model lifecycle data?",
        "correct_answer": "get_model_lifecycle(url='https://us-south.ml.cloud.ibm.com', model_id='ibm/granite-13b-instruct-v2')",
        "correct_answer_document_ids": ["fm_helpers.html"],
    },
    {
        "question": "What is path to ModelInference class?",
        "correct_answer": "ibm_watsonx_ai.foundation_models.inference.ModelInference",
        "correct_answer_document_ids": ["fm_model_inference.html"],
    },
    {
        "question": "What is method for get model inference details?",
        "correct_answer": "get_details()",
        "correct_answer_document_ids": ["fm_model_inference.html"],
    },
]

Upload the benchmark testing data to the bucket as a `json` file.

In [16]:
import json

test_filename = "benchmarking_data_ibm_watson_ai.json"

if not os.path.isfile(test_filename):
    with open(test_filename, "w") as json_file:
        json.dump(benchmarking_data_IBM_page_content, json_file, indent=4)

test_asset_details = client.data_assets.create(
    name=test_filename, file_path=test_filename
)

test_asset_id = client.data_assets.get_id(test_asset_details)
test_asset_id

Creating data asset...
SUCCESS


'01a08592-f9b4-741f-a400-14532bd87666'

Define connection information to the testing data.

In [17]:
from ibm_watsonx_ai.helpers import DataConnection

test_data_references = [DataConnection(data_asset_id=test_asset_id)]

### Set up connectivity information to Milvus

<b>This notebook focuses on a self-managed Milvus cluster using <a href="https://cloud.ibm.com/docs/watsonxdata?topic=watsonxdata-adding-milvus-service" target="_blank" rel="noopener no referrer">IBM watsonx.data.</a></b>

The following cell retrieves the Milvus username, password, host, and port from the environment (if available) and prompts you to provide them manually in case of failure.

You can provide a connection asset ID to read all required connection data from it. Before doing so, make sure that a connection asset was created in your space.

In [18]:
import getpass
import os

milvus_connection_id = input(
    "Provide connection asset ID in your space. Skip this, if you wish to type credentials by hand and hit enter: "
)

if not milvus_connection_id:
    try:
        username = os.environ["MILVUS_USER"]
    except KeyError:
        username = input("Please enter your Milvus user name and hit enter: ")

    try:
        password = os.environ["MILVUS_PASSWORD"]
    except KeyError:
        password = getpass.getpass("Please enter your Milvus password and hit enter: ")

    try:
        host = os.environ["MILVUS_HOST"]
    except KeyError:
        host = input("Please enter your Milvus hostname and hit enter: ")

    try:
        port = os.environ["MILVUS_PORT"]
    except KeyError:
        port = input("Please enter your Milvus port number and hit enter: ")

    try:
        ssl = os.environ["MILVUS_SSL"]
    except:
        ssl = bool(
            input(
                "Please enter ('y'/anything) if your Milvus instance has SSL enabled. Skip if it is not: "
            )
        )

    # Create connection
    milvus_data_source_type_id = client.connections.get_datasource_type_uid_by_name(
        "milvus"
    )
    details = client.connections.create(
        {
            client.connections.ConfigurationMetaNames.NAME: "Milvus Connection",
            client.connections.ConfigurationMetaNames.DESCRIPTION: "Connection created by the sample notebook",
            client.connections.ConfigurationMetaNames.DATASOURCE_TYPE: milvus_data_source_type_id,
            client.connections.ConfigurationMetaNames.PROPERTIES: {
                "host": host,
                "port": port,
                "username": username,
                "password": password,
                "ssl": ssl,
            },
        }
    )

    milvus_connection_id = client.connections.get_id(details)

Creating connections...
SUCCESS


Define connection information to vector store references.

In [19]:
vector_store_references = [DataConnection(connection_asset_id=milvus_connection_id)]

### RAG Optimizer configuration

Provide the input information for the AutoAI RAG optimizer:
- `name` - experiment name
- `description` - experiment description
- `max_number_of_rag_patterns` - maximum number of RAG patterns to create
- `optimization_metrics` - target optimization metrics

In [20]:
from ibm_watsonx_ai.experiment import AutoAI
from ibm_watsonx_ai.foundation_models.schema import (
    AutoAIRAGGenerationConfig,
    AutoAIRAGModelConfig,
    AutoAIRAGRetrievalConfig,
)

experiment = AutoAI(
    credentials=credentials,
    space_id=space_id,
)

retrieval_config = AutoAIRAGRetrievalConfig(
    method="window",
    number_of_chunks=6,
    window_size=2,
)

chunking_config = {"method": "recursive", "chunk_size": 512, "chunk_overlap": 128}

foundation_model = AutoAIRAGModelConfig(
    model_id="mistralai/mistral-small-3-1-24b-instruct-2503",
)

generation_config = AutoAIRAGGenerationConfig(
    foundation_models=[foundation_model],
)

rag_optimizer = experiment.rag_optimizer(
    name="AutoAI RAG test - sample notebook",
    description="Experiment run in sample notebook",
    chunking=[chunking_config],
    retrieval=[retrieval_config],
    generation=generation_config,
    embedding_model=["ibm/slate-125m-english-rtrvr-v2"],
    max_number_of_rag_patterns=4,
    optimization_metrics=[AutoAI.RAGMetrics.ANSWER_CORRECTNESS],
)

To retrieve the configuration parameters, use `get_params()`.

In [21]:
rag_optimizer.get_params()

{'name': 'AutoAI RAG test - sample notebook',
 'description': 'Experiment run in sample notebook',
 'chunking': [{'method': 'recursive',
   'chunk_size': 512,
   'chunk_overlap': 128}],
 'max_number_of_rag_patterns': 4,
 'optimization_metrics': ['answer_correctness'],
 'generation': {'foundation_models': [{'model_id': 'mistralai/mistral-small-3-1-24b-instruct-2503'}]},
 'retrieval': [{'method': 'window', 'number_of_chunks': 6, 'window_size': 2}]}

<a id="Run-the-RAG-Experiment"></a>
## Run the RAG Experiment

Call the `run()` method to trigger the AutoAI RAG experiment. Choose one of two modes: 

- To use the **interactive mode** (synchronous job), specify `background_mode=False` 
- To use the **background mode** (asynchronous job), specify `background_mode=True`

In [22]:
run_details = rag_optimizer.run(
    input_data_references=input_data_references,
    test_data_references=test_data_references,
    vector_store_references=vector_store_references,
    background_mode=False,
)



##############################################

Running 'e1ec40cc-75af-45fd-83b4-1a0261abb351'

##############################################


pending.......
running.........................................
completed
Training of 'e1ec40cc-75af-45fd-83b4-1a0261abb351' finished successfully.


To monitor the AutoAI RAG jobs in background mode, use the `get_run_status()` method.

In [23]:
rag_optimizer.get_run_status()

'completed'

<a id="Comparison-and-testing-of-RAG-Patterns"></a>
## Comparison and testing of RAG Patterns

You can list the trained patterns and information on evaluation metrics in the form of a Pandas DataFrame by calling the `summary()` method. Use the DataFrame to compare all discovered patterns and select the one you want for further testing.

In [24]:
summary = rag_optimizer.summary()
summary

,mean_answer_correctness,chunking.method,chunking.chunk_size,chunking.chunk_overlap,embeddings.model_id,vector_store.distance_metric,retrieval.method,retrieval.number_of_chunks,generation.model_id,agent.type
Pattern_Name,,,,,,,,,,
Pattern1,0.6429,recursive,512,128,ibm/slate-125m-english-rtrvr-v2,cosine,window,6,mistralai/mistral-small-3-1-24b-instruct-2503,sequential


Additionally, you can pass the `scoring` parameter to the summary method to filter RAG patterns, starting with the best.

In [25]:
summary = rag_optimizer.summary(scoring="faithfulness")

### Get the selected pattern

Get the RAGPattern object from the RAG Optimizer experiment. By default, the RAGPattern of the best pattern is returned.

In [26]:
best_pattern_name = summary.index.values[0]
print("Best pattern is:", best_pattern_name)

best_pattern = rag_optimizer.get_pattern()

Best pattern is: Pattern1


To retrieve the pattern details, use the `get_pattern_details` method.

In [27]:
rag_optimizer.get_pattern_details()

{'composition_steps': ['model_selection',
  'chunking',
  'embedding',
  'retrieval',
  'generation',
  'optimization'],
 'duration_seconds': 42,
 'location': {'evaluation_results': 'default_autoai_rag_out/e1ec40cc-75af-45fd-83b4-1a0261abb351/Pattern1/evaluation_results.json',
  'indexing_notebook': 'default_autoai_rag_out/e1ec40cc-75af-45fd-83b4-1a0261abb351/Pattern1/indexing_notebook.ipynb',
  'indexing_service_code': 'default_autoai_rag_out/e1ec40cc-75af-45fd-83b4-1a0261abb351/Pattern1/indexing_ai_service.gz',
  'indexing_service_metadata': 'default_autoai_rag_out/e1ec40cc-75af-45fd-83b4-1a0261abb351/Pattern1/indexing_service_metadata.json',
  'inference_notebook': 'default_autoai_rag_out/e1ec40cc-75af-45fd-83b4-1a0261abb351/Pattern1/inference_notebook.ipynb',
  'inference_service_code': 'default_autoai_rag_out/e1ec40cc-75af-45fd-83b4-1a0261abb351/Pattern1/inference_ai_service.gz',
  'inference_service_metadata': 'default_autoai_rag_out/e1ec40cc-75af-45fd-83b4-1a0261abb351/Pattern1/

Query the RAGPattern locally to test it.

In [28]:
from ibm_watsonx_ai.deployments import RuntimeContext

runtime_context = RuntimeContext(api_client=client)

context = RuntimeContext(
    api_client=client,
    request_payload_json={
        "messages": [
            {
                "role": "user",
                "content": "How to use new approach of providing credentials to APIClient?",
            }
        ]
    },
)

resp = best_pattern.inference_service(runtime_context)[0](context)

In [29]:
print(resp["body"]["choices"][0]["message"]["content"])

To use the new approach of providing credentials to `APIClient` in IBM WatsonX AI v1.0, you need to follow these steps, as indicated in the "New authorization" and "V1 Migration Guide" sections:

**New authorization**

```python
from ibm_watsonx_ai import APIClient
from ibm_watsonx_ai import Credentials

# Create a Credentials object with the required parameters
credentials = Credentials(
    url="https://us-south.ml.cloud.ibm.com",
    token="your_api_token",
)

# Initialize the APIClient with the Credentials object
client = APIClient(credentials)
```

**V1 Migration Guide**

```python
from ibm_watsonx_ai import APIClient
from ibm_watsonx_ai import Credentials

# Create a Credentials object with the required parameters
credentials = Credentials(
    url="https://us-south.ml.cloud.ibm.com",  # Replace with your specific URL if needed
    token="your_api_token",
)

# Initialize the APIClient with the Credentials object
client = APIClient(credentials)
```

In both cases, you need to repl

### Deploy the RAGPattern

To deploy the RAGPattern, store the defined RAG function and then create a deployed asset.

In [30]:
deployment_details = best_pattern.inference_service.deploy(
    name="AutoAI RAG deployment - ibm_watsonx_ai documentataion", space_id=space_id
)



######################################################################################

Synchronous deployment creation for id: '01a085a7-3f32-77b2-9ca5-b09d42c6dd94' started

######################################################################################


initializing
Note: online_url and serving_urls are deprecated and will be removed in a future release. Use inference instead.
...
ready


-----------------------------------------------------------------------------------------------
Successfully finished deployment creation, deployment_id='01a085a7-5f29-7536-904b-4da60dbd5d8b'
-----------------------------------------------------------------------------------------------




### Test the deployed function

The RAG service is now deployed in our space. To test the solution, run the cell below. Questions have to be provided in the payload. Their format is provided below.

In [31]:
deployment_id = client.deployments.get_id(deployment_details)

question = "How to add Task Credentials?"

payload = {"messages": [{"role": "user", "content": question}]}
score_response = client.deployments.run_ai_service(deployment_id, payload)

In [32]:
print(score_response["choices"][0]["message"]["content"])

To add Task Credentials on IBM watsonx.ai for IBM Cloud, you can follow the steps outlined below. The process is described in the document titled "IBM watsonx.ai for IBM Cloud" - so there's no need to look into other documents for this question.

1. **Initialize an APIClient object**:
   Before you can add Task Credentials, you need to initialize an APIClient object. You can do this by using the following code:
   ```python
   from ibm_watsonx_ai import APIClient
   client = APIClient(credentials, project_id=project_id)
   ```
   Alternatively, if you are using a space ID instead of a project ID, you can initialize the client like this:
   ```python
   client = APIClient(credentials, space_id=space_id)
   ```

2. **List available task credentials**:
Add Task Credentials consists of a few steps. First, you need to list the available Task Credentials to see what's already set up. You can do this using the `list` method:
   ```python
   client.task_credentials.list()
   ```
   often an em

<a id="Historical-runs"></a>
## Historical runs

In this section, you will learn how to work with historical RAG Optimizer jobs (runs).

To list historical runs, use the `list()` method and provide the `'rag_optimizer'` filter.

In [33]:
experiment.runs(filter="rag_optimizer").list()

,timestamp,run_id,state,auto_pipeline_optimizer name
0,2026-09-09T09:58:51.234Z,e1ec40cc-75af-45fd-83b4-1a0261abb351,completed,AutoAI RAG test - sample notebook


In [34]:
run_id = run_details["metadata"]["id"]
run_id

'e1ec40cc-75af-45fd-83b4-1a0261abb351'

### Get the executed optimizer's configuration parameters

In [35]:
experiment.runs.get_rag_params(run_id=run_id)

{'name': 'AutoAI RAG test - sample notebook',
 'description': 'Experiment run in sample notebook',
 'chunking': [{'chunk_overlap': 128,
   'chunk_size': 512,
   'method': 'recursive'}],
 'max_number_of_rag_patterns': 4,
 'generation': {'foundation_models': [{'model_id': 'mistralai/mistral-small-3-1-24b-instruct-2503'}]},
 'retrieval': [{'method': 'window', 'number_of_chunks': 6, 'window_size': 2}],
 'optimization_metrics': ['answer_correctness']}

### Get the historical rag_optimizer instance and training details

In [36]:
historical_opt = experiment.runs.get_rag_optimizer(run_id)

### List trained patterns for the selected optimizer

In [37]:
historical_opt.summary()

,mean_answer_correctness,chunking.method,chunking.chunk_size,chunking.chunk_overlap,embeddings.model_id,vector_store.distance_metric,retrieval.method,retrieval.number_of_chunks,generation.model_id,agent.type
Pattern_Name,,,,,,,,,,
Pattern1,0.6429,recursive,512,128,ibm/slate-125m-english-rtrvr-v2,cosine,window,6,mistralai/mistral-small-3-1-24b-instruct-2503,sequential


<a id="Cleanup"></a>
## Cleanup

To delete the current experiment, use the `cancel_run(hard_delete=True)` method.

**Warning:** Be careful: once you delete an experiment, you will no longer be able to refer to it.

In [38]:
rag_optimizer.cancel_run(hard_delete=True)

'SUCCESS'

To delete the deployment, use the `delete` method. 

**Warning:** If you keep the deployment active, it might lead to unnecessary consumption of Compute Unit Hours (CUHs).

In [39]:
client.deployments.delete(deployment_id)

'SUCCESS'

To clean up all of the created assets:
- experiments
- trainings
- pipelines
- model definitions
- models
- functions
- deployments

follow the steps in this sample [notebook](https://github.com/IBM/watson-machine-learning-samples/blob/master/cpd5.1/notebooks/python_sdk/instance-management/Machine%20Learning%20artifacts%20management.ipynb).

<a id="Summary-and-next-steps"></a>
## Summary and next steps

You successfully completed this notebook!

You learned how to use `ibm-watsonx-ai` to run AutoAI RAG experiments. 

Check out our _<a href="https://ibm.github.io/watsonx-ai-python-sdk/samples.html" target="_blank" rel="noopener no referrer">Online Documentation</a>_ for more samples, tutorials, documentation, how-tos, and blog posts. 

### Authors and Maintainers

**Mateusz Szewczyk (Former)**, Software Engineer at IBM watsonx.ai

**Paweł Kocur**, Software Engineer at IBM watsonx.ai

**Rafał Chrzanowski**, Software Engineer at IBM watsonx.ai

Copyright © 2025-2026 IBM. This notebook and its source code are released under the terms of the MIT License.